In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier


#1. Load data and choose features

In [11]:
df = pd.read_csv("../../data/H2_cleaned.csv")
LEAKY = [
    "ReservationStatus",        # the outcome itself
    "ReservationStatusDate",    # date of the outcome
    #"AssignedRoomType",         # assigned at/near check-in
    "BookingChanges",           # grows until check-in or cancellation
    "RequiredCarParkingSpaces", # can be added after booking
    #"TotalOfSpecialRequests",   # can be added after booking
    #"ArrivalDateYear",          # not usable for future bookings
]
NUM = ["LeadTime", "ArrivalDateWeekNumber", "ArrivalDateDayOfMonth", "StaysInWeekendNights",
       "StaysInWeekNights", "Adults", "Children", "Babies", "PreviousCancellations","TotalOfSpecialRequests","ArrivalDateYear",
       "PreviousBookingsNotCanceled", "DaysInWaitingList", "ADR"]
CAT = ["ArrivalDateMonth", "Meal", "Country", "MarketSegment", "DistributionChannel",
       "ReservedRoomType", "DepositType", "CustomerType","Agent"]
BINARY = ["IsRepeatedGuest", "HasCompany"]  

X = df.drop(columns=["IsCanceled"] + LEAKY)[NUM + CAT + BINARY].copy()
y = df["IsCanceled"]

for c in CAT:
    X[c] = X[c].astype(str).str.strip()
X["Agent"] = X["Agent"].str.replace(r"\.0$", "", regex=True)
X = X.replace({"nan": np.nan, "Undefined": np.nan, "NULL": np.nan})

2.Train/test split

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print("train:", X_train.shape, "| test:", X_test.shape)

train: (42474, 25) | test: (10619, 25)


3. Imputation

In [13]:
imputer = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUM),
    ("cat", SimpleImputer(strategy="constant", fill_value="missing"), CAT),
], remainder="passthrough", verbose_feature_names_out=False)

4.Encoding

In [14]:
encoder = ColumnTransformer([
    ("ohe", OneHotEncoder(handle_unknown="infrequent_if_exist",
                          min_frequency=30, sparse_output=False), CAT),
], remainder="passthrough", verbose_feature_names_out=False)

5. Scaling

In [15]:
scaler = ColumnTransformer([
    ("scale", StandardScaler(), NUM),
], remainder="passthrough", verbose_feature_names_out=False)

In [ ]:
prep = Pipeline([("impute", imputer), ("encode", encoder), ("scale", scaler)])
prep.set_output(transform="pandas")
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, random_state=42),
    "decision_tree": DecisionTreeClassifier(random_state=42),
    "k_nearest_neighbors": KNeighborsClassifier(n_jobs=-1),
    "xgboost": XGBClassifier(random_state=42, n_jobs=-1, eval_metric='logloss'),
    "random_forest": RandomForestClassifier(random_state=42, n_jobs=-1),
}
cv = StratifiedKFold(5, shuffle=True, random_state=42)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

results = {}
for name, model in models.items():
    pipe = Pipeline([("prep", clone(prep)), ("model", model)])
    scores = cross_validate(pipe, X_train, y_train, cv=cv, scoring=scoring)
    results[name] = {m: scores[f"test_{m}"].mean() for m in scoring}
    pipe.fit(X_train, y_train)

pd.DataFrame(results).T.round(4)

,accuracy,precision,recall,f1,roc_auc
logistic_regression,0.7690,0.6641,0.4758,0.5544,0.8158
decision_tree,0.7642,0.6082,0.6157,0.6119,0.7234
k_nearest_neighbors,0.7627,0.6217,0.5471,0.5820,0.7813
gaussian_nb,0.4997,0.3650,0.8833,0.5163,0.7183
xgboost,0.8143,0.7230,0.6240,0.6698,0.8800
random_forest,0.8208,0.7448,0.6185,0.6758,0.8780
